## BUILDING THE RAG SYSTEM - (vector DB)

- taking the cleaned hotel, attraction, transport, scam and mall data and converting it into a searchable vector database(embeddings) that the AI can query in real time. 
- the AI searches my created (csv) database for relevant information, then hands that information to the AI to generate a grounded and accurate response. The AI only answers based on what I give it.

### EMBEDDINGS
-  So for two pieces of text that mean the same thing, they'll have similar vectors even if they use different words. So a question like "cheap hotels near the beach in Diani" will correctly retrieve hotels described as "affordable beachfront resort" because their meanings are close in vector space not just keyword matches (Synonims).
- 187 documents embedded and stored in ChromaDB to power the travel assistant.


In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import os

BASE = r'C:\Users\User\Desktop\travel-africa-rag' # variable - raw string - as written

c:\Users\User\Desktop\travel-africa-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
hotels = pd.read_csv(f'{BASE}/data/cleaned/hotels_cleaned.csv')
attractions = pd.read_csv(f'{BASE}/data/cleaned/attractions_cleaned.csv')
transport = pd.read_csv(f'{BASE}/data/cleaned/transport_cleaned.csv')
scams = pd.read_csv(f'{BASE}/data/cleaned/scams_cleaned.csv')
malls = pd.read_csv(f'{BASE}/data/cleaned/malls_cleaned.csv')

print("All data loaded")
print("Total records:", len(hotels) + len(attractions) + len(transport) + len(scams) + len(malls))

All data loaded
Total records: 187


In [3]:
print("Loading embedding model... this may take a minute")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully")

Loading embedding model... this may take a minute


c:\Users\User\Desktop\travel-africa-rag\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4408.07it/s]


Model loaded successfully


In [4]:
chroma_client = chromadb.PersistentClient(path=f'{BASE}/chroma_db')

collection = chroma_client.get_or_create_collection(
    name="travel_africa",
    metadata={"hnsw:space": "cosine"}
)
print("ChromaDB collection ready")

ChromaDB collection ready


In [5]:
texts = hotels['combined_text'].tolist()
ids = [f"hotel_{i}" for i in range(len(texts))]
metadatas = hotels[['hotel_name','location','country','category','price_range','rating','source_url']].to_dict('records')

embeddings = model.encode(texts, show_progress_bar=True).tolist()

collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metadatas)
print(f"Added {len(texts)} hotels to ChromaDB")

Batches: 100%|██████████| 3/3 [00:02<00:00,  1.20it/s]

Added 95 hotels to ChromaDB


In [6]:
texts = attractions['combined_text'].tolist()
ids = [f"attraction_{i}" for i in range(len(texts))]
metadatas = attractions[['name','location','country','type','price_range']].to_dict('records')

embeddings = model.encode(texts, show_progress_bar=True).tolist()
collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metadatas)
print(f"Added {len(texts)} attractions to ChromaDB")

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.48it/s]

Added 25 attractions to ChromaDB


In [7]:
texts = transport['combined_text'].tolist()
ids = [f"transport_{i}" for i in range(len(texts))]
metadatas = transport[['name','city','country','transport_type','approximate_cost']].to_dict('records')

embeddings = model.encode(texts, show_progress_bar=True).tolist()
collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metadatas)
print(f"Added {len(texts)} transport options to ChromaDB")

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s]

Added 29 transport options to ChromaDB


In [8]:
texts = scams['combined_text'].tolist()
ids = [f"scam_{i}" for i in range(len(texts))]
metadatas = scams[['scam_name','location','country']].to_dict('records')

embeddings = model.encode(texts, show_progress_bar=True).tolist()
collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metadatas)
print(f"Added {len(texts)} scam warnings to ChromaDB")

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.22it/s]

Added 20 scam warnings to ChromaDB


In [9]:
texts = malls['combined_text'].tolist()
ids = [f"mall_{i}" for i in range(len(texts))]
metadatas = malls[['name','location','country','opening_hours']].to_dict('records')

embeddings = model.encode(texts, show_progress_bar=True).tolist()
collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metadatas)
print(f"Added {len(texts)} malls to ChromaDB")

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]

Added 18 malls to ChromaDB


In [10]:
print("Total documents in ChromaDB:", collection.count())

Total documents in ChromaDB: 187


In [11]:
query = "affordable hotels in Diani near the beach"
query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print("Query:", query)
print("\nTop results:")
for i, doc in enumerate(results['documents'][0]):
    print(f"\n--- Result {i+1} ---")
    print(doc[:300])

Query: affordable hotels in Diani near the beach

Top results:

--- Result 1 ---
Hotel: Almanara Luxury Villas. Location: Diani, Kenya. Category: Luxury. Description: Exclusive boutique resort with private pool villas steps from Diani Beach. Price range: 300-800 USD. Amenities: WiFi, Private Pool, Beach, Restaurant, Bar, Spa, Butler Service. Room types: Villa, Suite. Rating: 4.9

--- Result 2 ---
Hotel: Diani Reef Beach Resort. Location: Diani, Kenya. Category: Beach Resort. Description: Sprawling beachfront resort with water sports and multiple restaurants. Price range: 100-350 USD. Amenities: WiFi, Pool, Beach, Restaurant, Bar, Spa, Water Sports, Kids Club. Room types: Standard, Deluxe, Su

--- Result 3 ---
Hotel: Leopard Beach Resort. Location: Diani, Kenya. Category: Beach Resort. Description: Long established resort with lush gardens and beautiful beachfront. Price range: 100-300 USD. Amenities: WiFi, Pool, Beach, Restaurant, Bar, Spa, Water Sports. Room types: Standard, Deluxe, S